In [1]:
# 如果不显示图形化界面，首先安装ipywidgets
# %pip install ipywidgets pandas

In [2]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import subprocess
import os
import re

# --- 配置区域 ---
DNAWORKS_PATH = "./dnaworks"
INP_DIR = "./inpfiles"
RESULT_DIR = "./results"

# 确保文件夹存在
for d in [INP_DIR, RESULT_DIR]:
    if not os.path.exists(d):
        os.makedirs(d)

In [ ]:
# --- 预设密码子表 ---
# SerAGC会被错误识别为Asn，需要避免
# ProCCA仅会被唯一一个密码子识别，需要避免，ProCCU,ProCCC会造成+1位核糖体移码，应在重组基因中彻底避免
CODON_TABLES = {
    "E. coli": """CODOn
Gly     GGG     40359.00      11.39      0.16
Gly     GGA     34894.00       9.85      0.13
Gly     GGT     89915.00      25.37      0.35
Gly     GGC     94608.00      26.70      0.36
Glu     GAG     66665.00      18.81      0.33
Glu     GAA    137748.00      38.87      0.67
Asp     GAT    116164.00      32.78      0.63
Asp     GAC     67865.00      19.15      0.37
Val     GTG     85263.00      24.06      0.34
Val     GTA     41283.00      11.65      0.17
Val     GTT     70627.00      19.93      0.29
Val     GTC     50417.00      14.23      0.20
Ala     GCG    104293.00      29.43      0.32
Ala     GCA     75329.00      21.26      0.23
Ala     GCT     60787.00      17.15      0.19
Ala     GCC     85138.00      24.03      0.26
Arg     AGG      7966.00       2.25      0.04
Arg     AGA     13784.00       3.89      0.07
Ser     AGT     35966.00      10.15      0.16
Ser     AGC     53286.00      15.04      0.24
Lys     AAG     45133.00      12.74      0.26
Lys     AAA    125351.00      35.37      0.74
Asn     AAT     75086.00      21.19      0.50
Asn     AAC     75334.00      21.26      0.50
Met     ATG     92952.00      26.23      1.00
Ile     ATA     25982.00       7.33      0.12
Ile     ATT    105218.00      29.69      0.49
Ile     ATC     83118.00      23.46      0.39
Thr     ACG     48560.00      13.70      0.25
Thr     ACA     34483.00       9.73      0.17
Thr     ACT     37430.00      10.56      0.19
Thr     ACC     77023.00      21.74      0.39
Trp     TGG     48949.00      13.81      1.00
End     TGA      3616.00       1.02      0.31
Cys     TGT     18601.00       5.25      0.46
Cys     TGC     21434.00       6.05      0.54
End     TAG       978.00       0.28      0.08
End     TAA      7024.00       1.98      0.60
Tyr     TAT     62750.00      17.71      0.59
Tyr     TAC     43034.00      12.14      0.41
Leu     TTG     45581.00      12.86      0.13
Leu     TTA     51320.00      14.48      0.14
Phe     TTT     78743.00      22.22      0.58
Phe     TTC     56591.00      15.97      0.42
Ser     TCG     29993.00       8.46      0.13
Ser     TCA     32814.00       9.26      0.15
Ser     TCT     37586.00      10.61      0.17
Ser     TCC     32586.00       9.20      0.15
Arg     CGG     21391.00       6.04      0.11
Arg     CGA     13645.00       3.85      0.07
Arg     CGT     70009.00      19.76      0.36
Arg     CGC     68569.00      19.35      0.35
Gln     CAG    100346.00      28.32      0.66
Gln     CAA     51275.00      14.47      0.34
His     CAT     44633.00      12.60      0.58
His     CAC     32678.00       9.22      0.42
Leu     CTG    168885.00      47.66      0.47
Leu     CTA     15275.00       4.31      0.04
Leu     CTT     42704.00      12.05      0.12
Leu     CTC     35873.00      10.12      0.10
Pro     CCG     72450.00      20.44      0.49
Pro     CCA     30515.00       8.61      0.21
Pro     CCT     26805.00       7.56      0.18
Pro     CCC     19008.00       5.36      0.13
//""",
    "P. pastoris": """CODOn
Gly     GGG       468.00       5.76      0.00
Gly     GGA      1550.00      19.06      0.00
Gly     GGT      2075.00      25.52      0.00
Gly     GGC       655.00       8.06      0.00
Glu     GAG      2360.00      29.03      0.00
Glu     GAA      3043.00      37.43      0.00
Asp     GAT      2899.00      35.66      0.00
Asp     GAC      2103.00      25.87      0.00
Val     GTG       998.00      12.28      0.00
Val     GTA       804.00       9.89      0.00
Val     GTT      2188.00      26.91      0.00
Val     GTC      1210.00      14.88      0.00
Ala     GCG       314.00       3.86      0.00
Ala     GCA      1228.00      15.10      0.00
Ala     GCT      2351.00      28.92      0.00
Ala     GCC      1348.00      16.58      0.00
Arg     AGG       539.00       6.63      0.00
Arg     AGA      1634.00      20.10      0.00
Ser     AGT      1020.00      12.55      0.00
Ser     AGC       621.00       7.64      0.00
Lys     AAG      2748.00      33.80      0.00
Lys     AAA      2433.00      29.93      0.00
Asn     AAT      2038.00      25.07      0.00
Asn     AAC      2168.00      26.67      0.00
Met     ATG      1517.00      18.66      0.00
Ile     ATA       906.00      11.14      0.00
Ile     ATT      2532.00      31.14      0.00
Ile     ATC      1580.00      19.43      0.00
Thr     ACG       491.00       6.04      0.00
Thr     ACA      1118.00      13.75      0.00
Thr     ACT      1820.00      22.39      0.00
Thr     ACC      1175.00      14.45      0.00
Trp     TGG       834.00      10.26      0.00
End     TGA        27.00       0.33      0.00
Cys     TGT       626.00       7.70      0.00
Cys     TGC       356.00       4.38      0.00
End     TAG        40.00       0.49      0.00
End     TAA        69.00       0.85      0.00
Tyr     TAT      1300.00      15.99      0.00
Tyr     TAC      1473.00      18.12      0.00
Leu     TTG      2562.00      31.51      0.00
Leu     TTA      1265.00      15.56      0.00
Phe     TTT      1963.00      24.14      0.00
Phe     TTC      1675.00      20.60      0.00
Ser     TCG       598.00       7.36      0.00
Ser     TCA      1234.00      15.18      0.00
Ser     TCT      1983.00      24.39      0.00
Ser     TCC      1344.00      16.53      0.00
Arg     CGG       158.00       1.94      0.00
Arg     CGA       340.00       4.18      0.00
Arg     CGT       564.00       6.94      0.00
Arg     CGC       175.00       2.15      0.00
Gln     CAG      1323.00      16.27      0.00
Gln     CAA      2069.00      25.45      0.00
His     CAT       960.00      11.81      0.00
His     CAC       737.00       9.07      0.00
Leu     CTG      1215.00      14.94      0.00
Leu     CTA       873.00      10.74      0.00
Leu     CTT      1289.00      15.85      0.00
Leu     CTC       620.00       7.63      0.00
Pro     CCG       320.00       3.94      0.00
Pro     CCA      1540.00      18.94      0.00
Pro     CCT      1282.00      15.77      0.00
Pro     CCC       553.00       6.80      0.00
//""",
    "B. subtilis": """CODOn
Gly     GGG      9094.00     11.15      0.00
Gly     GGA     17743.00     21.76      0.00
Gly     GGT     10566.00     12.96      0.00
Gly     GGC     18967.00     23.26      0.00
Glu     GAG     18429.00     22.60      0.00
Glu     GAA     39217.00     48.09      0.00
Asp     GAT     27108.00     33.24      0.00
Asp     GAC     15485.00     18.99      0.00
Val     GTG     14105.00     17.30      0.00
Val     GTA     10638.00     13.05      0.00
Val     GTT     15193.00     18.63      0.00
Val     GTC     14067.00     17.25      0.00
Ala     GCG     16176.00     19.84      0.00  
Ala     GCA     17205.00     21.10      0.00
Ala     GCT     15150.00     18.58      0.00
Ala     GCC     13433.00     16.47      0.00
Arg     AGG      3341.00      4.10      0.00
Arg     AGA      8561.00     10.50      0.00
Ser     AGT      5577.00      6.84      0.00
Ser     AGC     11766.00     14.43      0.00
Lys     AAG     16997.00     20.84      0.00
Lys     AAA     39449.00     48.38      0.00
Asn     AAT     18702.00     22.93      0.00
Asn     AAC     14522.00     17.81      0.00
Met     ATG     21424.00     26.27      0.00
Ile     ATA      7960.00      9.76      0.00
Ile     ATT     29487.00     36.16      0.00
Ile     ATC     22167.00     27.18      0.00
Thr     ACG     12110.00     14.85      0.00
Thr     ACA     17642.00     21.63      0.00
Thr     ACT      7082.00      8.68      0.00
Thr     ACC      7342.00      9.00      0.00
Trp     TGG      8765.00     10.75      0.00
End     TGA       621.00      0.76      0.00
Cys     TGT      2939.00      3.60      0.00
Cys     TGC      3472.00      4.26      0.00
End     TAG       381.00      0.47      0.00
End     TAA      1562.00      1.92      0.00
Tyr     TAT     18967.00     23.26      0.00
Tyr     TAC     10290.00     12.62      0.00
Leu     TTG     12914.00     15.84      0.00
Leu     TTA     16167.00     19.83      0.00
Phe     TTT     24450.00     29.98      0.00
Phe     TTC     11677.00     14.32      0.00
Ser     TCG      5266.00      6.46      0.00
Ser     TCA     11874.00     14.56      0.00
Ser     TCT     10320.00     12.66      0.00
Ser     TCC      6766.00      8.30      0.00
Arg     CGG      5664.00      6.95      0.00
Arg     CGA      3537.00      4.34      0.00
Arg     CGT      5903.00      7.24      0.00
Arg     CGC      6720.00      8.24      0.00
Gln     CAG     15089.00     18.50      0.00
Gln     CAA     16620.00     20.38      0.00
His     CAT     12832.00     15.74      0.00
His     CAC      6141.00      7.53      0.00
Leu     CTG     18757.00     23.00      0.00
Leu     CTA      3975.00      4.87      0.00
Leu     CTT     17772.00     21.79      0.00
Leu     CTC      8697.00     10.67      0.00
Pro     CCG     13316.00     16.33      0.00
Pro     CCA      5796.00      7.11      0.00
Pro     CCT      8640.00     10.60      0.00
Pro     CCC      2850.00      3.50      0.00
//"""
}

In [ ]:
# --- UI 组件 ---
style = {'description_width': 'initial'}

ui_title = widgets.Text(value='INS_HUMAN', description='任务名称:', style=style)
ui_mode = widgets.Dropdown(options=[('蛋白质序列 (Protein)', 'PROTein'), ('核酸序列 (Nucleotide)', 'NUCLeotide')], value='PROTein', description='输入类型:', style=style)
ui_melting = widgets.IntSlider(value=62, min=50, max=80, description='第二轮Tm(℃):', style=style)
ui_len_high = widgets.BoundedIntText(value=57, min=45, max=59, description='引物最长长度 (nt):', style=style)
ui_len_low = widgets.BoundedIntText(value=50, min=45, max=59, description='引物最短长度 (nt):', style=style)
ui_codon = widgets.Dropdown(options=list(CODON_TABLES.keys()), value='E. coli', description='密码子优化物种:', style=style)
ui_frequency = widgets.IntSlider(value=20, min=5, max=30, description='最低密码子频率 (%):', style=style)
ui_seq = widgets.Textarea(
    value='FVNQHLCGSHLVEALYLVCGERGFFYTPKTRREAEDLQVGQVELGGGPGAGSLQPLALEGSLQKRGIVEQCCTSICSLYQLENYCN',
    description='序列:',
    layout={'height': '150px', 'width': '90%'},
    style=style
)

btn_run = widgets.Button(description='🚀 生成并运行 DNAWorks', button_style='primary', layout={'width': 'max-content'})
output = widgets.Output()

# --- 逻辑处理 ---

def process_and_validate_sequence(raw_input, mode):
    """
    1. 深度清洗：移除数字、空格、特殊符号
    2. 合法性校验：根据 mode (PROTein/NUCLeotide) 判定字符
    3. 误操作拦截：防止在蛋白模式下误贴 DNA
    4. 格式化：返回 DNAWorks 要求的带行号格式
    """
    # --- 1. 深度清洗 ---
    # 移除非字母字符并转大写 (一步到位替代 re.sub(r'[\d\s]', ...))
    clean_seq = re.sub(r'[^a-zA-Z]', '', raw_input).upper()
    
    if not clean_seq:
        raise ValueError("❌ 错误：输入序列不能为空或不包含有效字母！")

    # --- 2. 合法性校验 ---
    if mode == 'PROTein':
        valid_chars = set("ACDEFGHIKLMNPQRSTVWY")
        label = "蛋白质"
        
        # 启发式拦截：核酸误贴检查
        nt_count = sum(1 for c in clean_seq if c in "ATCG")
        if len(clean_seq) > 10 and (nt_count / len(clean_seq)) > 0.9:
            raise ValueError("⚠️ 风险拦截：检测到当前极像是【核酸序列】，但您选择了【蛋白质模式】！")
    else:
        valid_chars = set("ATCG")
        label = "核酸"

    # 检查非法字符
    invalid_found = [c for c in clean_seq if c not in valid_chars]
    if invalid_found:
        illegal_str = " ".join(set(invalid_found))
        raise ValueError(f"❌ 错误：{label}模式下包含非法字符: {illegal_str}")

    # --- 3. 格式化 (DNAWorks 标准格式) ---
    lines = []
    for i in range(0, len(clean_seq), 60):
        # 这里的格式：前缀空格 + 起始位点 + 序列
        lines.append(f"      {i+1:<3} {clean_seq[i:i+60]}")
    
    formatted_str = "\n".join(lines)
    
    return formatted_str, clean_seq

def update_tm_info(change):
    # change['new'] 会获取 ui_melting 改变后的新值
    new_tm = change['new'] - 4
    ui_first_round_tm.value = f"第一轮Tm(℃): {new_tm}"

# 绑定观察者模式：监听 value 属性的变化
ui_melting.observe(update_tm_info, names='value')

def run_dnaworks(b):
    with output:
        clear_output()
        
        try:
            # 一键完成清洗、校验和格式化
            formatted_seq, clean_seq = process_and_validate_sequence(ui_seq.value, ui_mode.value)
            
            # 定义路径
            name = ui_title.value.strip().replace(' ', '_')
            if ui_mode.value == 'PROTein':
                inp_path = os.path.join(INP_DIR, f"{name}_pro.inp")
                log_path = os.path.join(RESULT_DIR, f"{name}_pro.txt")
            else:
                inp_path = os.path.join(INP_DIR, f"{name}_nuc.inp")
                log_path = os.path.join(RESULT_DIR, f"{name}_nuc.txt")
            
            # 动态准备参数
            codon_section = CODON_TABLES[ui_codon.value] if ui_mode.value == 'PROTein' else ""

            # 构造内容 (注意这里直接引用 formatted_seq)
            inp_content = f"""title "{ui_title.value}"
melting low {ui_melting.value}
length low {ui_len_low.value} high {ui_len_high.value}
frequency threshold {ui_frequency.value}
logfile "{log_path}"

PATTern
   BsaI GGTCTC
   BspQI GCTCTTC
   SacI GAGCTC
   SalI GTCGAC
   BglII AGATCT
   Esp3I CGTCTC
//

{codon_section}

{ui_mode.value}
{formatted_seq}
//
"""
            # 根据模式（PROTein / NUCLeotide）准备不同的内容段
            #if ui_mode.value == 'PROTein':
            #    inp_path = os.path.join(INP_DIR, f"{name}_pro.inp")
            #    log_path = os.path.join(RESULT_DIR, f"{name}_pro.txt")
            #else:
            #    inp_path = os.path.join(INP_DIR, f"{name}_nuc.inp")
            #    log_path = os.path.join(RESULT_DIR, f"{name}_nuc.txt")
            # 写入文件
            with open(inp_path, 'w') as f:
                f.write(inp_content)
            
            print(f"✅ .inp 文件已保存至: {inp_path}")
            print(f"⏳ 正在启动 DNAWorks 调用 (实时输出)...\n")
        except ValueError as e:
            print(str(e)) # 直接显示上面函数抛出的错误信息
        except Exception as e:
            print(f"❌ 系统错误: {str(e)}")
        
        # --- 修改后的实时调用逻辑 ---
        try:
            # 使用 Popen 启动子进程，并将标准错误重定向到标准输出
            process = subprocess.Popen(
                [DNAWORKS_PATH, inp_path],
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
                bufsize=1  # 行缓冲，确保能实时读取到每一行
            )

            # 实时读取 stdout 中的每一行并立即打印到 Output 组件
            for line in iter(process.stdout.readline, ""):
                print(line, end="")  # line 自带换行符，所以用 end=""

            process.stdout.close()
            return_code = process.wait()

            if return_code == 0:
                print("\n" + "="*30)
                print("✅ DNAWorks 运行完成！")
                
                # 实时读取并显示结果文件中的内容
                if os.path.exists(log_path):
                    print(f"\n📄 最终优化结果 ({log_path}):")
                    print("-" * 30)
                    with open(log_path, 'r') as res_file:
                        print(res_file.read())
            else:
                print(f"\n❌ 程序异常退出，退出码: {return_code}")

        except FileNotFoundError:
            print(f"❌ 错误: 在路径 '{DNAWORKS_PATH}' 找不到可执行文件。")
        except Exception as e:
            print(f"❌ 系统错误: {str(e)}")

# 绑定按钮事件保持不变
btn_run.on_click(run_dnaworks)

# --- 布局显示 ---
ui_first_round_tm = widgets.HTML(
    value=f"第一轮Tm(℃): {ui_melting.value - 4}",
    style={'description_width': 'initial'},
)

input_box = widgets.VBox([
    widgets.HTML("<h3>🧬 图形化DNAWorks</h3>"),
    widgets.HBox([ui_title]),
    widgets.HBox([ui_mode]),
    widgets.HBox([ui_codon]),
    widgets.HBox([ui_frequency]),
    widgets.HBox([ui_first_round_tm]),
    widgets.HBox([ui_melting]),
    widgets.HBox([ui_len_low, ui_len_high]),
    ui_seq,
    btn_run
])

display(input_box, output)

Output()

In [12]:
import pandas as pd
import re
import os
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# --- 后处理 UI 组件 ---
style = {'description_width': 'initial'}

# 同步主程序的任务名
default_name = ui_title.value if 'ui_title' in globals() else "DNA_Project"

ui_export_name = widgets.Text(value=default_name, description='任务名称:', style=style)
ui_trial_id = widgets.BoundedIntText(value=8, min=1, max=100, step=1, description='导出 Trial #:', style=style)
btn_extract = widgets.Button(description='🔍 提取并预览引物', button_style='info', icon='search', layout={'width': '200px'})
out_msg = widgets.Output()

def on_extract_clicked(b):
    with out_msg:
        clear_output()
        name = ui_export_name.value.strip().replace(' ', '_')
        trial_num = ui_trial_id.value
        
        # 路径配置：根据模式匹配文件名后缀
        RESULT_DIR = "./results"
        suffix = "_pro.txt" if ui_mode.value == 'PROTein' else "_nuc.txt"
        log_path = os.path.join(RESULT_DIR, f"{name}{suffix}")
        
        if not os.path.exists(log_path):
            print(f"❌ 错误：未找到文件 {log_path}。")
            return

        try:
            with open(log_path, 'r') as f:
                content = f.read()

            # --- 正则解析逻辑 ---
            # 1. 定位 Trial 区域
            search_pattern = rf"PARAMETERS FOR TRIAL\s+{trial_num}\b(.*?)(?:PARAMETERS FOR TRIAL|$)"
            trial_section = re.search(search_pattern, content, re.DOTALL)

            if not trial_section:
                print(f"❌ 错误：在结果文件中未找到 TRIAL {trial_num}")
                return

            section_text = trial_section.group(1)

            # 2. 匹配引物行
            primer_pattern = r"(\w+),([ATCG]+),(\d+)nt"
            matches = re.findall(primer_pattern, section_text)

            if not matches:
                print(f"❌ 解析失败：在该 Trial 区域内未识别到符合格式的引物列表。")
                return

            # --- 3. 构造 DataFrame 并直接显示 ---
            data = []
            for p_name, p_seq, p_len in matches:
                data.append({
                    "Primer Name": p_name, 
                    "Sequence": p_seq, 
                    "Length (nt)": p_len
                })
            
            df = pd.DataFrame(data)
            
            print(f"✅ 成功从 {os.path.basename(log_path)} 提取 {len(matches)} 条引物")
            print("👇 请直接鼠标框选下方表格内容，复制后订购引物:")
            print("-" * 50)
            
            # 使用 HTML 渲染，index=False 彻底去掉左侧的 0,1,2,3...
            display(HTML(df.to_html(index=False)))

        except Exception as e:
            print(f"❌ 运行时发生错误: {str(e)}")

btn_extract.on_click(on_extract_clicked)

# --- 显示界面 ---
display(widgets.VBox([
    widgets.HTML("<h4>🛠️ DNAWorks 引物实时预览工具</h4>"),
    widgets.HBox([ui_export_name, ui_trial_id]),
    btn_extract,
    out_msg
]))